# Entraînement d'un modèle de régression linéaire avec MLflow

Ce notebook entraîne un modèle de régression linéaire simple et l'enregistre dans MLflow.

In [ ]:
# Installation des dépendances (si nécessaire)
# !pip install mlflow scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import mlflow
import mlflow.sklearn

## 1. Configuration de MLflow

In [ ]:
# Configuration du tracking URI (local par défaut)
# ATTENTION : il faut accéder à un registry distant pour rendre disponible votre modèle
mlflow.set_tracking_uri("http://localhost:5000")

# Nom de l'expérience
mlflow.set_experiment("linear_regression_experiment")

## 2. Génération de données synthétiques

In [ ]:
# Génération de données synthétiques pour la régression
np.random.seed(42)

# 1000 échantillons, 5 features
n_samples = 1000
n_features = 5

X = np.random.randn(n_samples, n_features)
# y = 3*x1 + 2*x2 - x3 + 0.5*x4 + x5 + bruit
coefficients = np.array([3, 2, -1, 0.5, 1])
y = X @ coefficients + np.random.randn(n_samples) * 0.5

print(f"Forme de X: {X.shape}")
print(f"Forme de y: {y.shape}")

## 3. Division des données

In [ ]:
# Division en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Taille ensemble d'entraînement: {X_train.shape[0]}")
print(f"Taille ensemble de test: {X_test.shape[0]}")

## 4. Entraînement du modèle avec MLflow

In [ ]:
# Démarrage d'un run MLflow
with mlflow.start_run(run_name="linear_regression_v1") as run:
    # Création et entraînement du modèle
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Prédictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calcul des métriques
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Log des paramètres
    mlflow.log_param("n_features", n_features)
    mlflow.log_param("n_samples", n_samples)
    mlflow.log_param("test_size", 0.2)
    
    # Log des métriques
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    
    # Enregistrement du modèle
    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="linear_regression_model"
    )
    
    # Affichage des résultats
    print("\n=== Résultats ===")
    print(f"Train MSE: {train_mse:.4f}")
    print(f"Test MSE: {test_mse:.4f}")
    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²: {test_r2:.4f}")
    print(f"\nRun ID: {run.info.run_id}")
    print(f"Model URI: runs:/{run.info.run_id}/model")

## 5. Test de chargement du modèle

In [ ]:
# Chargement du modèle depuis MLflow
loaded_model = mlflow.sklearn.load_model(f"runs:/{run.info.run_id}/model")

# Test de prédiction
test_sample = X_test[:5]
predictions = loaded_model.predict(test_sample)

print("\n=== Test de prédiction ===")
print(f"Prédictions: {predictions}")
print(f"Vraies valeurs: {y_test[:5]}")